In [6]:
!pip install sentence_transformers

     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ---------------------------------------- 41.5/41.5 kB 2.0 MB/s eta 0:00:00
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
    --------------------------------------- 10.2/596.4 kB ? eta -:--:--
   -- ------------------------------------ 41.0/596.4 kB 495.5 kB/s eta 0:00:02
   ------ -------------------------------- 92.2/596.4 kB 880.9 kB/s eta 0:00:01
   ------------- -------------------------- 204.8/596.4 kB 1.4 MB/s eta 0:00:01
   ------------------- -------------------- 286.7/596.4 kB 1.4 MB/s eta 0:00:01
   --------------------- ------------------ 317.4/596.4 kB 1.2 MB/s eta 0:00:01
   ------------------------ --------------- 368.6/596.4 kB 1.2 MB/s eta 0:00:01
   -----

In [43]:
pip install --upgrade tqdm


     ---------------------------------------- 0.0/57.4 kB ? eta -:--:--
     ------- -------------------------------- 10.2/57.4 kB ? eta -:--:--
     ------------- ------------------------ 20.5/57.4 kB 108.9 kB/s eta 0:00:01
     -------------------- ----------------- 30.7/57.4 kB 145.2 kB/s eta 0:00:01
     --------------------------------- ---- 51.2/57.4 kB 200.8 kB/s eta 0:00:01
     -------------------------------------- 57.4/57.4 kB 188.5 kB/s eta 0:00:00
   ---------------------------------------- 0.0/78.3 kB ? eta -:--:--
   -------------------- ------------------- 41.0/78.3 kB 653.6 kB/s eta 0:00:01
   ---------------------------------------- 78.3/78.3 kB 862.0 kB/s eta 0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.65.0
    Uninstalling tqdm-4.65.0:
      Successfully uninstalled tqdm-4.65.0


## 1. Import

In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.preprocessing import StandardScaler, OneHotEncoder, MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

In [2]:
import warnings
from tqdm import TqdmWarning

warnings.filterwarnings("ignore", category=TqdmWarning)

## 1. Data loading

In [3]:
with open('hotels_preprocessed.pkl', 'rb') as file:
    hotels_preprocessed = pickle.load(file)

df_model = pd.DataFrame(hotels_preprocessed)

In [4]:
print(df_model.shape)
df_model.head()

(2470, 72)


,id,name,category_name,category_clean,hotel_stars,hotel_stars_for_model,pets_policy,has_pets_info,rooms_count_original,rooms_count,...,beach_services_count,beach_type,beach_type_has_info,beach_type_count,hotel_type,hotel_type_has_info,hotel_type_count,internet,internet_has_info,internet_count
0,23209,Gleam Collection,Special Category,special,NaN,unknown,unknown,0,60.0,60,...,0,[],0,0,[одно здание],1,1,[wi-fi в номере],1,1
1,23213,Eras Alanya,3*,star_hotel,3.0,3,unknown,0,0.0,120,...,0,[],0,0,[],0,0,[],0,0
2,23214,Club Dizalya,4*,star_hotel,4.0,4,unknown,0,331.0,331,...,1,"[галька, песок крупный]",1,2,[отельный комплекс/большая территория],1,1,"[wi-fi, wi-fi в номере]",1,2
3,23215,Hatipoglu Beach,3*,star_hotel,3.0,3,unknown,0,70.0,70,...,1,[песок жёлтый],1,1,[одно здание],1,1,[wi-fi],1,1
4,23216,Seven Seas Palmeras Bay,5*,star_hotel,5.0,5,unknown,0,324.0,324,...,1,[песок жёлтый],1,1,[отельный комплекс/большая территория],1,1,"[wi-fi, wi-fi в номере]",1,2


In [5]:
print('ID duplicates:', df_model['id'].duplicated().sum())
print('Missing IDs:', df_model['id'].isna().sum())
print('Missing names:', df_model['name'].isna().sum())

ID duplicates: 0
Missing IDs: 0
Missing names: 0


In [6]:
df_model[['id', 'name', 'canonical_text']].head()

,id,name,canonical_text
0,23209,Gleam Collection,hotel name: Gleam Collection. category: specia...
1,23213,Eras Alanya,hotel name: Eras Alanya. category: star_hotel....
2,23214,Club Dizalya,hotel name: Club Dizalya. category: star_hotel...
3,23215,Hatipoglu Beach,hotel name: Hatipoglu Beach. category: star_ho...
4,23216,Seven Seas Palmeras Bay,hotel name: Seven Seas Palmeras Bay. category:...


## 3. Feature-based representation


In [7]:
categorical_cols = [
    'category_clean',
    'hotel_stars_for_model',
    'pets_policy',
    'geo_cluster'
]

numeric_cols = [
    'rooms_count_log',
    'rooms_count_missing'
]

In [8]:
# List of columns with service groups
service_cols = [
    'common_services',
    'parking',
    'room_amenities',
    'special_rooms',
    'food_services',
    'transport',
    'laundry_services',
    'entertainment',
    'health_and_beauty',
    'pool',
    'beach_line',
    'sports',
    'business_services',
    'children_services',
    'beach_services',
    'beach_type',
    'hotel_type',
    'internet'
]

### 3.1 Transformation

#### 3.1.1 OneHotEncoder

In [9]:
cat_matrix = pd.get_dummies(
    df_model[categorical_cols],
    columns=categorical_cols,
    dummy_na=False
)

cat_matrix.shape

(2470, 33)

#### 3.1.2 StandardScaler

In [10]:
scaler = StandardScaler()

num_matrix = scaler.fit_transform(df_model[numeric_cols])

num_matrix.shape

(2470, 2)

#### 3.1.3 MultiLabelBinarizer

In [11]:
service_matrices = []
service_feature_names = []

mlb_dict = {}

for col in service_cols:
    mlb = MultiLabelBinarizer()
    
    col_matrix = mlb.fit_transform(df_model[col])
    
    service_matrices.append(col_matrix)
    
    col_feature_names = [col + '__' + item for item in mlb.classes_]
    service_feature_names.extend(col_feature_names)
    
    mlb_dict[col] = mlb

### 3.2 Final matrix

In [12]:
service_matrix = np.hstack(service_matrices)

service_matrix.shape

(2470, 178)

In [13]:
feature_matrix = np.hstack([
    cat_matrix.values,
    num_matrix,
    service_matrix
])

feature_matrix.shape

(2470, 213)

In [14]:
feature_names = (
    list(cat_matrix.columns) +
    numeric_cols +
    service_feature_names
)

print('Number of feature names:', len(feature_names))
print('Feature matrix columns:', feature_matrix.shape[1])
print('Check:', len(feature_names) == feature_matrix.shape[1])

Number of feature names: 213
Feature matrix columns: 213
Check: True


### 3.3 Calculating similarity

In [15]:
feature_similarity_matrix = cosine_similarity(feature_matrix)

feature_similarity_matrix.shape

(2470, 2470)

### 3.4 Saving artifacts

In [16]:
with open('feature_matrix.pkl', 'wb') as file:
    pickle.dump(feature_matrix, file)

with open('feature_similarity_matrix.pkl', 'wb') as file:
    pickle.dump(feature_similarity_matrix, file)

with open('feature_names.pkl', 'wb') as file:
    pickle.dump(feature_names, file)

with open('mlb_dict.pkl', 'wb') as file:
    pickle.dump(mlb_dict, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

## 4. Embedding-based representation

In [17]:
texts = df_model['canonical_text'].fillna('').tolist()

print('Number of texts:', len(texts))

Number of texts: 2470


In [18]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### 4.1 Creating text embeddings

In [19]:
texts = df_model['canonical_text'].fillna('').tolist()

text_embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=False,
    normalize_embeddings=True
)

text_embeddings.shape

(2470, 384)

### 4.2 Calculating similarity

In [20]:
embedding_similarity_matrix = cosine_similarity(text_embeddings)

embedding_similarity_matrix.shape

(2470, 2470)

### 4.3 Saving

In [21]:
with open('text_embeddings.pkl', 'wb') as file:
    pickle.dump(text_embeddings, file)

with open('embedding_similarity_matrix.pkl', 'wb') as file:
    pickle.dump(embedding_similarity_matrix, file)

## 5. Combined similarity

### 5.1 Preparing

In [22]:
print('Feature-based similarity:')
print('min:', feature_similarity_matrix.min())
print('max:', feature_similarity_matrix.max())
print('mean:', feature_similarity_matrix.mean())

print()

print('Embedding-based similarity:')
print('min:', embedding_similarity_matrix.min())
print('max:', embedding_similarity_matrix.max())
print('mean:', embedding_similarity_matrix.mean())

Feature-based similarity:
min: -0.14992183353451843
max: 1.0000000000000018
mean: 0.3924483949916099

Embedding-based similarity:
min: 0.33333272
max: 1.0000007
mean: 0.7877454


In [23]:
def min_max_normalize(matrix):
    matrix_min = matrix.min()
    matrix_max = matrix.max()
    
    normalized_matrix = (matrix - matrix_min) / (matrix_max - matrix_min)
    
    return normalized_matrix

In [24]:
feature_similarity_norm = min_max_normalize(feature_similarity_matrix)
embedding_similarity_norm = min_max_normalize(embedding_similarity_matrix)

In [25]:
print('Feature normalized:')
print('min:', feature_similarity_norm.min())
print('max:', feature_similarity_norm.max())

print()

print('Embedding normalized:')
print('min:', embedding_similarity_norm.min())
print('max:', embedding_similarity_norm.max())

Feature normalized:
min: 0.0
max: 1.0

Embedding normalized:
min: 0.0
max: 1.0


### 5.2 Final similarity matrix

In [26]:
feature_weight = 0.3
embedding_weight = 0.7

In [27]:
final_similarity_matrix = (
    feature_weight * feature_similarity_norm +
    embedding_weight * embedding_similarity_norm
)

final_similarity_matrix.shape

(2470, 2470)

## 6. Save modeling artifacts

In [28]:
with open('final_similarity_matrix.pkl', 'wb') as file:
    pickle.dump(final_similarity_matrix, file)

similarity_weights = {
    'feature_weight': feature_weight,
    'embedding_weight': embedding_weight
}

with open('similarity_weights.pkl', 'wb') as file:
    pickle.dump(similarity_weights, file)

In [29]:
with open('feature_similarity_norm.pkl', 'wb') as file:
    pickle.dump(feature_similarity_norm, file)

with open('embedding_similarity_norm.pkl', 'wb') as file:
    pickle.dump(embedding_similarity_norm, file)